# 🧪 Praktikum 07 – Vektordatenbanken & Profi-RAG
**Applied Generative AI – NLP | Sommersemester 2026**

> 🎯 **Lernziele:** Persistente Vektorspeicher verwalten · Hybride Retrieval-Strategien · RAG-Antworten auf Faktentreue (Faithfulness) prüfen

⏱️ **Dauer:** 90 Minuten

In [1]:
import importlib
import os
import shutil
import subprocess
import sys
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "chromadb": ("chromadb", "1.1.1"),
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}


def run_install(specs):
    if shutil.which("uv") is None:
        raise RuntimeError("uv is not installed. Install uv and rerun the setup cell.")

    command = ["uv", "pip", "install", "--python", sys.executable, *specs]
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix) or bool(os.environ.get("VIRTUAL_ENV"))
    if not in_venv:
        command.append("--system")

    print("$", " ".join(command))
    subprocess.check_call(command)


missing_specs = [
    f"{install_name}=={version}"
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import chromadb
import ollama
import requests


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


def model_is_available(requested_name, available_names):
    candidates = {requested_name, f"{requested_name}:latest"}
    return any(candidate in available_names for candidate in candidates)


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if not IN_COLAB:
        raise RuntimeError("Ollama is not installed locally. Install Ollama and rerun the setup cell.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    ollama_log = open("/tmp/ollama-notebook.log", "a", encoding="utf-8")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
for model_name in [MODEL, EMBED_MODEL]:
    subprocess.check_call(["ollama", "pull", model_name], env=env)

payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
missing_models = [
    model_name
    for model_name in [MODEL, EMBED_MODEL]
    if not model_is_available(model_name, available_models)
]
if missing_models:
    raise RuntimeError(f"Missing local Ollama models: {', '.join(missing_models)}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Models:", MODEL, ",", EMBED_MODEL)
print("Available local models:", ", ".join(available_models))

$ uv pip install --python /Users/joschkakersting/Nextcloud/vl/Folien/Praktikum/.venv-py313/bin/python chromadb==1.1.1


Using Python 3.13.8 environment at: .venv-py313
Resolved 78 packages in 242ms
Installed 38 packages in 46ms
 + backoff==2.2.1
 + bcrypt==5.0.0
 + build==1.4.3
 + chromadb==1.1.1
 + distro==1.9.0
 + durationpy==0.10
 + flatbuffers==25.12.19
 + googleapis-common-protos==1.74.0
 + grpcio==1.80.0
 + httptools==0.7.1
 + importlib-metadata==8.7.1
 + importlib-resources==7.1.0
 + kubernetes==35.0.0
 + mmh3==5.2.1
 + oauthlib==3.3.1
 + onnxruntime==1.24.4
 + opentelemetry-api==1.41.0
 + opentelemetry-exporter-otlp-proto-common==1.41.0
 + opentelemetry-exporter-otlp-proto-grpc==1.41.0
 + opentelemetry-proto==1.41.0
 + opentelemetry-sdk==1.41.0
 + opentelemetry-semantic-conventions==0.62b0
 + orjson==3.11.8
 + overrides==7.7.0
 + posthog==5.4.0
 + protobuf==6.33.6
 + pybase64==1.4.3
 + pypika==0.51.1
 + pyproject-hooks==1.2.0
 + python-dotenv==1.2.2
 + requests-oauthlib==2.0.0
 + tenacity==9.1.4
 + uvicorn==0.44.0
 + uvloop==0.22.1
 + watchfiles==1.1.1
 + websocket-client==1.9.0
 + websockets==1

Runtime: Local/Jupyter
Models: qwen3.5:0.8b , nomic-embed-text
Available local models: nomic-embed-text:latest, qwen3.5:0.8b, devstral-small-2:latest, devstral-small-2:24b


pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


## Teil 1 – Advanced ChromaDB Management ⏱️ 30 min
Wir nutzen Metadaten-Filter und persistente Indexierung.

In [2]:
client = chromadb.PersistentClient(path="./rag_storage")
collection = client.get_or_create_collection(name="docs")

texts = [
    "Python ist eine interpretierte Hochsprache.",
    "C++ ist eine kompilierte, maschinennahe Sprache.",
    "Rust bietet Speichersicherheit ohne Garbage Collector.",
    "Java nutzt eine Virtual Machine (JVM)."
]

def embed(t): return ollama.embeddings(model=EMBED_MODEL, prompt=t)["embedding"]

if collection.count() == 0:
    collection.add(
        documents=texts,
        embeddings=[embed(t) for t in texts],
        metadatas=[{"lang_type": "high"}, {"lang_type": "low"}, {"lang_type": "safe"}, {"lang_type": "vm"}],
        ids=[f"id{i}" for i in range(len(texts))]
    )
print(f"Anzahl Dokumente in DB: {collection.count()}")

Anzahl Dokumente in DB: 4


## Teil 2 – Hybrides Retrieval ⏱️ 30 min
Nur Vektorsuche reicht oft nicht. Wir kombinieren sie mit Keyword-Matching (simuliert).

In [6]:
import re


def hybrid_search(query, top_k=2):
    candidate_count = max(top_k, collection.count())
    v_res = collection.query(query_embeddings=[embed(query)], n_results=candidate_count)

    keywords = [token for token in re.findall(r"\w+", query.lower()) if len(token) > 2]
    stopwords = {"was", "ist", "der", "die", "das", "ein", "eine", "und"}
    content_terms = [token for token in keywords if token not in stopwords]
    if not content_terms:
        content_terms = keywords

    final_docs = []
    for rank, doc in enumerate(v_res["documents"][0]):
        doc_lower = doc.lower()
        lexical_score = sum(1 for term in content_terms if term in doc_lower)
        vector_bonus = max(0, candidate_count - rank)
        total_score = lexical_score * 10 + vector_bonus
        final_docs.append((total_score, doc))

    final_docs.sort(key=lambda item: item[0], reverse=True)
    return [doc for _, doc in final_docs[:top_k]]

print("Hybride Suche für 'JVM Sprache':", hybrid_search("JVM Sprache"))

Hybride Suche für 'JVM Sprache': ['Java nutzt eine Virtual Machine (JVM).', 'C++ ist eine kompilierte, maschinennahe Sprache.']


## Teil 3 – RAG Faithfulness & Self-Correction ⏱️ 30 min
Wir lassen das Modell prüfen, ob die Antwort wirklich im Kontext steht.

In [7]:
def rag_with_check(query):
    context_docs = hybrid_search(query, top_k=1)
    context = "\n".join(context_docs)
    gen_prompt = (
        "Beantworte die Frage ausschliesslich mit Informationen aus dem Kontext. "
        "Wenn die Information im Kontext fehlt, antworte exakt mit: Unbekannt. "
        "Antworte in einem kurzen deutschen Satz.\n\n"
        f"Kontext:\n{context}\n\nFrage: {query}"
    )
    answer = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": gen_prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 60},
        keep_alive="10m",
    )["message"]["content"].strip()
    if not answer:
        raise RuntimeError("Das Modell hat keine Antwort erzeugt.")

    check_prompt = (
        "Pruefe, ob die Antwort vollstaendig durch den Kontext gedeckt ist. "
        "Antworte exakt mit JA oder NEIN.\n\n"
        f"Kontext:\n{context}\n\nAntwort: {answer}"
    )
    valid = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": check_prompt}],
        think=False,
        options={"temperature": 0, "num_predict": 8, "stop": ["\n"]},
        keep_alive="10m",
    )["message"]["content"].strip()

    return answer, valid

ans, status = rag_with_check("Was ist Rust?")
print(f"Antwort: {ans}\nVerifiziert: {status}")

Antwort: Rust ist ein speicherhaltiger Sprachen, der ohne Garbage Collector Speichersicherheit bietet.
Verifiziert: JA


## 🧩 Aufgaben
1. **Re-Ranking:** Implementiere ein einfaches Re-Ranking, das Dokumente bevorzugt, die kürzer als 50 Zeichen sind.
2. **Error Handling:** Was passiert, wenn kein Dokument gefunden wird? Baue einen Fallback-Prompt.
3. **Metadaten-Explosion:** Füge 100 Test-Dokumente hinzu und filtere nach `lang_type`. Messe die Zeitdifferenz zur ungefilterten Suche.